# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [2]:
import pandas as pd

# 1. Load data
url1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
url2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
url3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

df1, df2, df3 = pd.read_csv(url1), pd.read_csv(url2), pd.read_csv(url3)

# 2. Align and Combine
df3.rename(columns={'State': 'ST'}, inplace=True)
df = pd.concat([df1, df2, df3], axis=0, ignore_index=True)

# 3. Clean Names and Types
df.columns = [col.lower().replace(" ", "_") for col in df.columns]
df.rename(columns={'st': 'state'}, inplace=True)

# Clean CLV (Remove % and convert to numeric)
df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'].astype(str).str.replace('%', ''), errors='coerce')

# Clean Complaints (Extract middle value from 1/X/00)
df['number_of_open_complaints'] = pd.to_numeric(df['number_of_open_complaints'].apply(lambda x: x.split('/')[1] if isinstance(x, str) else x), errors='coerce')

# 4. Standardize Categorical Values (The simplified version of Point #7)
clean_map = {
    "gender": {"Femal": "F", "female": "F", "Male": "M"},
    "state": {"AZ": "Arizona", "Cali": "California", "WA": "Washington"},
    "education": {"Bachelors": "Bachelor"}
}
for col, mapping in clean_map.items():
    df[col] = df[col].replace(mapping)

# 5. Handle Nulls and Duplicates
# Fill numbers with median, fill text with the most frequent value (mode), then drop duplicates
df = df.fillna(df.median(numeric_only=True))
df = df.apply(lambda x: x.fillna(x.mode()[0]) if x.dtype == "O" else x)
df = df.drop_duplicates().reset_index(drop=True)

# 6. Final Polish: Convert numbers to integers
num_cols = ['income', 'monthly_premium_auto', 'number_of_open_complaints', 'total_claim_amount']
df[num_cols] = df[num_cols].astype(int)

print(f"Cleaned Dataframe Shape: {df.shape}")
df.head()

Cleaned Dataframe Shape: (9135, 12)


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount,gender
0,RB50392,Washington,F,Master,7.714877e+03,0,1000,0,Personal Auto,Four-Door Car,2,F
1,QZ44356,Arizona,F,Bachelor,6.979536e+05,0,94,0,Personal Auto,Four-Door Car,1131,F
2,AI49188,Nevada,F,Bachelor,1.288743e+06,48767,108,0,Personal Auto,Two-Door Car,566,F
3,WW63253,California,M,Bachelor,7.645862e+05,0,106,0,Corporate Auto,SUV,529,F
4,GA49547,Washington,M,High School or Below,5.363077e+05,36357,68,0,Personal Auto,Four-Door Car,17,F


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [3]:
import pandas as pd

# Load the comprehensive dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df = pd.read_csv(url)

# Display columns to see the new features (Response, EmploymentStatus, etc.)
print(f"Columns available: {df.columns.tolist()}")
df.head()

# Create a pivot table to compare groups
pivot_claims = df.pivot_table(index='education', columns='gender', values='total_claim_amount', aggfunc='mean')

print("Average Total Claim Amount by Education and Gender:")
print(pivot_claims)

# Grouping by Sales Channel and calculating multiple statistics
channel_analysis = df.groupby('sales_channel')['customer_lifetime_value'].agg(['mean', 'median', 'count'])

# Sorting to see the most profitable channel at the top
channel_analysis = channel_analysis.sort_values(by='mean', ascending=False)

print("\nSales Channel Analysis:")
print(channel_analysis)

Columns available: ['unnamed:_0', 'customer', 'state', 'customer_lifetime_value', 'response', 'coverage', 'education', 'effective_to_date', 'employmentstatus', 'gender', 'income', 'location_code', 'marital_status', 'monthly_premium_auto', 'months_since_last_claim', 'months_since_policy_inception', 'number_of_open_complaints', 'number_of_policies', 'policy_type', 'policy', 'renew_offer_type', 'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size', 'vehicle_type', 'month']
Average Total Claim Amount by Education and Gender:
gender                         F           M
education                                   
Bachelor              415.083016  440.472063
College               389.436584  458.548390
Doctor                332.498824  341.660257
High School or Below  459.650066  519.500247
Master                349.908049  359.090827

Sales Channel Analysis:
                      mean       median  count
sales_channel                                 
Call Center    8110.36

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [4]:
# 1. Create the pivot table for total revenue per sales channel
revenue_pivot = df.pivot_table(index='sales_channel', values='income', aggfunc='sum').round(2)

# Rename column for clarity
revenue_pivot.columns = ['total_revenue']

print("Summary Table: Total Revenue by Sales Channel")
print(revenue_pivot)

# 2. Create pivot table for Average CLV
clv_pivot = df.pivot_table(index='education', columns='gender', values='customer_lifetime_value', aggfunc='mean').round(2)

print("\nAverage CLV per Gender and Education Level:")
print(clv_pivot)

# A. Convert date column to datetime
df['effective_to_date'] = pd.to_datetime(df['effective_to_date'])

# B. Extract the month (as name or number)
df['month'] = df['effective_to_date'].dt.month_name()

# C. Create a summary table in Long Format
# Group by Month and Policy Type and sum the complaints
complaints_long = df.groupby(['month', 'policy_type'])['number_of_open_complaints'].sum().reset_index()

# Sort by month and complaints to make it readable
print("\nComplaints by Policy Type and Month (Long Format):")
print(complaints_long)

Summary Table: Total Revenue by Sales Channel
               total_revenue
sales_channel               
Agent              152490152
Branch             113775608
Call Center         81055004
Web                 62200103

Average CLV per Gender and Education Level:
gender                      F        M
education                             
Bachelor              7874.27  7703.60
College               7748.82  8052.46
Doctor                7328.51  7415.33
High School or Below  8675.22  8149.69
Master                8157.05  8168.83

Complaints by Policy Type and Month (Long Format):
      month     policy_type  number_of_open_complaints
0  February  Corporate Auto                 385.208135
1  February   Personal Auto                1453.684441
2  February    Special Auto                  95.226817
3   January  Corporate Auto                 443.434952
4   January   Personal Auto                1727.605722
5   January    Special Auto                  87.074049
